<a href="https://colab.research.google.com/github/SchmetterlingIII/physics-maths/blob/main/emergence_of_seizures__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%autosave 50

Autosaving every 50 seconds


# Modelling the emergence of seizures in the brain

I want to model out an abstraction of the propagation of seizures in the brain to try and develop my intuition for this network and as an introduction to computational neuroscience.
I have already looked, briefly, at mathematical percolation (https://youtu.be/a-767WnbaCQ?si=73i8cgjbz-zR8Q7W) and this may inform my intuitions.

## Abstraction Approaches
Here are the current ideas that I have:
- Agent-based model (ABM) of neurotransmitters between pre- and post- synaptic nerves (connecting axon terminals to dendrites) to see how differences in concentrations of glutamate and GABA ($\gamma$-aminobutryate acid) affects the action potetials and global likelihood for excitation in neurons. *This would be close-up and hyper-local; the outputted probability from this could then be the input for a grid graph that could be rendered in a similar way to Bernoucilli percolation.*
- A more global design where the network has naturally, randomly occurring electrical impulses that propagates through the network (indicated by a highlighted node). The control variables are the E/I balance (and later mutations within certain cells) so that the global patterns are showed in this model.

**With these models, I need to re-evaluate why it is necessary to develop them:**
- Educational & introductory into the more technical aspects of epilepsy.
- Could collaborate with others on the other aspects of this tool (and reach out to agencies ... why?)


*I have decided to progress with the network model as it will be the easiest to complete in a short amount of time and still has a high roof: I will be able to add additional complexity (with probabilities for excitation, speed of propagation and UI controls).* **The ABM can come later with more detailed understanding and can be animated in a similar manner to - for example - Kurzgesagt's abstractions of biological complexity.**

## Attempt I: Global Model
*This is my first attempt at first quantifying what it is I want to model, what success criteria there would be and what limitations of this would be.*

On a 2D lattice grid, there are nodes and edges: nodes here will represent the soma and dendrites; edges will represent axon and axon terminal.

There is a low, background likelihood for the random excitation of any given node - which would change colour. If a cell is excited, there is a given probability that its neighbouring cells would also light up (an arbitrary probability for this).

I would like to model this dynamically and so, using the slider in Colab, I will be able to control the number of steps in this (writing an iterative function as the function).

Parameters I wish to control within this:
- Size of the lattice ($n \times n$ grid)
- The number of steps taken already
- The background excitation probability
- The neighbouring excitation probability


#### Current Thoughts
I think that this will be a pretty weak abstraction of the process (as it is too abstract in this form), however it shows the bare-bones essence of electrical activity propagation in the brain.

In [18]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as ipy

In [28]:
n = 23
G = nx.grid_2d_graph(n, n)

# print(G.nodes()) # a tupled structure of all the nodes

nx.set_node_attributes(G, False, 'active')       # whether electrical signal has been passed through it
nx.set_node_attributes(G, 'blue', 'colour')      # a visual representation of the signal
nx.set_node_attributes(G, 0, 'timer')            # localised time attribute
nx.set_node_attributes(G, False, 'refractory')   # activating the refractory period for cells after having activated the timer
nx.set_node_attributes(G, 3, 'refractory_timer') # associated refractory timer in which nothing can happen to it

In [24]:
def neuron_abstraction_I(graph_object, background_excitation=0.1, neighbouring_excitation=0.5, activation_timer=5):
    """
    1. Check all activate cells without any changes and see if the timer has exceeded the limit and with the refractory period
    (refractory cells are effectively removed from this graph for this period e.g. available_neurons = [neuron['refractory'=False] for neuron in G])
    2. Apply the background activation
    3. Apply the neighbouring cell thing
    4. Update characteristics
    """

    nodes = graph_object.nodes()
    # it would be better to deal with these like a numpy array rather than calling the list like this
    # handling all in one go will help when this scales to 3D where there will be more connection to cater for

    for node in nodes:
        # 'refractory_neuron' countdown
        if nodes[node]['refractory']:
            nodes[node]['refractory_timer'] -= 1
            if nodes[node]['refractory_timer'] <= 0:
                nodes[node]['refractory'] = False
                nodes[node]['refractory_timer'] = 3  # reset

        # 'active_neuron' timer
        if nodes[node]['active']:
            nodes[node]['timer'] += 1
            if nodes[node]['timer'] >= activation_timer:
                nodes[node]['active'] = False
                nodes[node]['colour'] = 'blue'
                nodes[node]['refractory'] = True
                nodes[node]['timer'] = 0

    inactive_non_refractory_neurons = [node for node in nodes if not nodes[node]['refractory']] # returns non-refractory neurons - not sure whether inactive ones should be included yet

    """For the background probability, I am going to apply a Bernoulli mask as shown in the percolation video"""
    neuron_array = np.array(nodes)
    n_neurons = len(neuron_array)
    rand_prob = np.random.rand(n_neurons)
    selection_mask = np.where(rand_prob <= background_excitation)[0]
    activation_mask = neuron_array[selection_mask]

    for node_array in activation_mask:
        node_key = tuple(node_array) # only tuples are hashable
        nodes[node_key]['active'] = True
        nodes[node_key]['colour'] = 'red'
        nodes[node_key]['refractory'] = False
        nodes[node_key]['timer'] = 0

    '''
    For neighbouring activation, find all active cells and make a list of all of their neighbours (which aren't active themselves)
    Then apply the same Bernoulli mask (different values, same principle) and activate them

    The issue with the initial function was that the for loop would be such that the niehbouring neurons could activate recursively, rather than having a mechanism that would store it to know which
    '''

    all_activated_neurons = [node for node in nodes if nodes[node]['active']]

    # Use a set to collect the keys of the neighbours to activate,
    # without recursively activating neighbours in a single time step
    neighbours_to_activate = set()

    if all_activated_neurons:
        for neuron in all_activated_neurons:
            eligible_neighbours = [
                neigh for neigh in graph_object.neighbors(neuron)
                if not nodes[neigh]['active'] and not nodes[neigh]['refractory']
            ]

            if not eligible_neighbours:
                continue # different to pass

            eligible_neighbours_array = np.array(eligible_neighbours, dtype=object) # dtype=object makes the array act like a typical pythonic list
            n_eligible = len(eligible_neighbours_array)

            # applying the bernoulli mask (as usual)
            rand_prob = np.random.rand(n_eligible)
            bernoulli_mask = np.where(rand_prob <= neighbouring_excitation)[0]

            eligible_neighbour_keys = eligible_neighbours_array[bernoulli_mask]
            for key in eligible_neighbour_keys:
                neighbours_to_activate.add(tuple(key))  # Convert array to tuple

    for neighbour in neighbours_to_activate:
        nodes[neighbour]['active'] = True
        nodes[neighbour]['colour'] = 'red'
        nodes[neighbour]['refractory'] = False
        nodes[neighbour]['timer'] = 0

    pos = nx.kamada_kawai_layout(graph_object)
    colours = [nodes[node]['colour'] for node in graph_object.nodes()]
    nx.draw(graph_object, pos=pos, node_color=colours, node_size=25)
    #display(fig)

One note about the above code is that it will be computationally inefficient - even with the use of numpy to work through things as matrices rather than in a `for` loop.

I will use these static diagrams as a start but try to work out (**and actually work out, not just ask Claude**) how to make this process more efficient **as well as** evaluating how abstract this current model is from actual neuronal connections.

How easy will this be to move onto 3D?

### Visualisation Function
I want to have this dynamical system by viewed dynamically (and to see clearly how, in this simple model, changes to the global parameters will yield pretty different results).

In [29]:
ipy.interact(neuron_abstraction_I, graph_object=ipy.fixed(G), background_excitation=(0, 0.3, 0.01), neighbouring_excitation=(0.3, 1, 0.05), activation_timer=(0,10,1))

interactive(children=(FloatSlider(value=0.1, description='background_excitation', max=0.3, step=0.01), FloatSl…

<function __main__.neuron_abstraction_I(graph_object, background_excitation=0.1, neighbouring_excitation=0.5, activation_timer=5)>